In [ ]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/MedAssist AI/clean_190k_dataset.csv')

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_clean = df.copy()
# STEP 1: LABEL ENCODING
print("Translating diseases to numbers...")
encoder = LabelEncoder()
df_clean['target'] = encoder.fit_transform(df_clean['diseases'])
df_final = df_clean.drop(columns=['diseases'])

Translating diseases to numbers...


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("Preparing train/test split with rare disease preservation...")

# -------------------------------------------------------
# STEP 1: Identify rare and common diseases
# -------------------------------------------------------
class_counts = df_final['target'].value_counts()

rare_classes = class_counts[class_counts == 1].index
common_classes = class_counts[class_counts > 1].index

# -------------------------------------------------------
# STEP 2: Separate rare and common diseases
# -------------------------------------------------------
df_rare = df_final[df_final['target'].isin(rare_classes)]
df_common = df_final[df_final['target'].isin(common_classes)]

# -------------------------------------------------------
# STEP 3: Stratified train-test split on common diseases
# -------------------------------------------------------
X_common = df_common.drop(columns=['target'])
y_common = df_common['target']

X_train_common, X_test, y_train_common, y_test = train_test_split(
    X_common,
    y_common,
    test_size=0.2,
    random_state=42,
    stratify=y_common
)

# -------------------------------------------------------
# STEP 4: Add all rare diseases to the training set
# -------------------------------------------------------
X_rare = df_rare.drop(columns=['target'])
y_rare = df_rare['target']

X_train = pd.concat([X_train_common, X_rare], ignore_index=True)
y_train = pd.concat([y_train_common, y_rare], ignore_index=True)

# -------------------------------------------------------
# STEP 5: Downsample ONLY the training set
# -------------------------------------------------------
CAP = 600

train_df = X_train.copy()
train_df["target"] = y_train

train_df = (
    train_df
    .groupby("target", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), CAP), random_state=42))
    .reset_index(drop=True)
)

X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

# -------------------------------------------------------
# STEP 6: Print summary
# -------------------------------------------------------
print("\n========== DATA PREPARATION COMPLETE ==========")
print(f"Training rows : {len(X_train):,}")
print(f"Testing rows  : {len(X_test):,}")
print(f"Unique diseases in training : {y_train.nunique()}")
print(f"Maximum samples per disease : {y_train.value_counts().max()}")

print("\nTraining class distribution:")
print(y_train.value_counts().describe())

Preparing train/test split with rare disease preservation...


/tmp/ipykernel_1629/579435391.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), CAP), random_state=42))



========== DATA PREPARATION COMPLETE ==========
Training rows : 134,353
Testing rows  : 37,921
Unique diseases in training : 773
Maximum samples per disease : 600

Training class distribution:
count    773.000000
mean     173.807245
std      214.093245
min        1.000000
25%        8.000000
50%       62.000000
75%      252.000000
max      600.000000
Name: count, dtype: float64


In [ ]:
# Define the evaluation set for live tracking
eval_set = [(X_train, y_train), (X_test, y_test)]

In [ ]:
# ===============================
# Random Forest
# ===============================

import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("========== Training Random Forest ==========")

t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

acc_rf = accuracy_score(y_test, rf_preds) * 100
prec_rf = precision_score(y_test, rf_preds, average='weighted', zero_division=0) * 100
rec_rf = recall_score(y_test, rf_preds, average='weighted', zero_division=0) * 100
f1_rf = f1_score(y_test, rf_preds, average='weighted', zero_division=0) * 100

results = []

results.append({
    'Model': 'Random Forest',
    'Accuracy (%)': round(acc_rf, 2),
    'Precision (%)': round(prec_rf, 2),
    'Recall (%)': round(rec_rf, 2),
    'F1-Score (%)': round(f1_rf, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})

results_df = pd.DataFrame(results)

print("\n========== Random Forest Results ==========")
print(results_df.to_string(index=False))

In [ ]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print(sample_weights.min())
print(sample_weights.max())

NameError: name 'compute_sample_weight' is not defined

In [ ]:
# ===============================
# XGBoost
# ===============================

import xgboost as xgb
import time
from sklearn.utils.class_weight import compute_sample_weight

print("========== Training XGBoost ==========")

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

t0 = time.time()

xgb_model = xgb.XGBClassifier(
    tree_method="hist",
    device="cuda",

    random_state=42,

    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,

    subsample=0.8,
    colsample_bytree=0.8,

    eval_metric="mlogloss"
)

xgb_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=1
)

xgb_preds = xgb_model.predict(X_test)

acc_xgb = accuracy_score(y_test, xgb_preds) * 100
prec_xgb = precision_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100
rec_xgb = recall_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100
f1_xgb = f1_score(y_test, xgb_preds, average='weighted', zero_division=0) * 100

results = []

results.append({
    'Model': 'XGBoost',
    'Accuracy (%)': round(acc_xgb, 2),
    'Precision (%)': round(prec_xgb, 2),
    'Recall (%)': round(rec_xgb, 2),
    'F1-Score (%)': round(f1_xgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})

results_df = pd.DataFrame(results)

print("\n========== XGBoost Results ==========")
print(results_df.to_string(index=False))

========== Training XGBoost ==========
[0]	validation_0-mlogloss:5.86764
[1]	validation_0-mlogloss:5.23763
[2]	validation_0-mlogloss:4.74108
[3]	validation_0-mlogloss:4.34484
[4]	validation_0-mlogloss:4.02974
[5]	validation_0-mlogloss:3.77742
[6]	validation_0-mlogloss:3.56944
[7]	validation_0-mlogloss:3.38955
[8]	validation_0-mlogloss:3.23150
[9]	validation_0-mlogloss:3.08970
[10]	validation_0-mlogloss:2.96355
[11]	validation_0-mlogloss:2.84912
[12]	validation_0-mlogloss:2.74495
[13]	validation_0-mlogloss:2.64883
[14]	validation_0-mlogloss:2.55917
[15]	validation_0-mlogloss:2.47681
[16]	validation_0-mlogloss:2.39976
[17]	validation_0-mlogloss:2.32786
[18]	validation_0-mlogloss:2.26017
[19]	validation_0-mlogloss:2.19639
[20]	validation_0-mlogloss:2.13685
[21]	validation_0-mlogloss:2.08013
[22]	validation_0-mlogloss:2.02655
[23]	validation_0-mlogloss:1.97566
[24]	validation_0-mlogloss:1.92744
[25]	validation_0-mlogloss:1.88135
[26]	validation_0-mlogloss:1.83742
[27]	validation_0-mlogloss

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [14:20:11] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



========== XGBoost Results ==========
  Model  Accuracy (%)  Precision (%)  Recall (%)  F1-Score (%)  Time (Mins)
XGBoost         79.96          82.57       79.96          80.8        21.85


In [ ]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    top_k_accuracy_score
)
import pandas as pd

# ==========================
# Predictions & Probabilities
# ==========================

preds = lgb_model.predict(X_test)
probs = lgb_model.predict_proba(X_test)

# ==========================
# Macro F1
# ==========================

macro_f1 = f1_score(y_test, preds, average='macro')

# ==========================
# Top-3 Accuracy
# ==========================

top3 = top_k_accuracy_score(
    y_test,
    probs,
    k=3,
    labels=xgb_model.classes_
)

# ==========================
# Classification Report
# ==========================

report = classification_report(
    y_test,
    preds,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

# ==========================
# Rare Disease Analysis
# ==========================

# Count samples in the TEST SET
test_counts = y_test.value_counts()

# Diseases having 3 or fewer test samples
rare_classes = test_counts[test_counts <= 3].index

rare_metrics = report_df.loc[
    report_df.index.intersection(rare_classes.astype(str))
]

print("========== XGBoost Complete Evaluation ==========")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Top-3 Accuracy: {top3:.4f}")

print("\n--- 🔬 METRICS FOR RARE DISEASES (3 or fewer test samples) ---")

if len(rare_metrics) > 0:
    print(rare_metrics[['precision','recall','f1-score']].mean())
else:
    print("No rare diseases found in the test set.")

========== XGBoost Complete Evaluation ==========
Macro F1      : 0.4645
Top-3 Accuracy: 0.8631

--- 🔬 METRICS FOR RARE DISEASES (3 or fewer test samples) ---
precision    0.040335
recall       0.354528
f1-score     0.068101
dtype: float64


In [ ]:
import re

# Instantly strip all illegal JSON characters from column names
X_train = X_train.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))
X_test = X_test.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '_', x))

# Re-define eval_set so it uses the newly cleaned X_test
eval_set = [(X_train, y_train), (X_test, y_test)]

print("--- Column Names Cleaned for LightGBM! ---")

--- Column Names Cleaned for LightGBM! ---


In [ ]:
# ===============================
# LightGBM
# ===============================

import lightgbm as lgb
import time

print("========== Training LightGBM ==========")

eval_set = [(X_train, y_train), (X_test, y_test)]

t0 = time.time()

lgb_model = lgb.LGBMClassifier(
    random_state=42,
    objective="multiclass",

    n_estimators=300,
    max_depth=6,

    min_child_samples=1,

    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=eval_set,
    callbacks=[lgb.early_stopping(30, verbose=False)]
)

lgb_preds = lgb_model.predict(X_test)

acc_lgb = accuracy_score(y_test, lgb_preds) * 100
prec_lgb = precision_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
rec_lgb = recall_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100
f1_lgb = f1_score(y_test, lgb_preds, average='weighted', zero_division=0) * 100

results = []

results.append({
    'Model': 'LightGBM',
    'Accuracy (%)': round(acc_lgb, 2),
    'Precision (%)': round(prec_lgb, 2),
    'Recall (%)': round(rec_lgb, 2),
    'F1-Score (%)': round(f1_lgb, 2),
    'Time (Mins)': round((time.time() - t0) / 60, 2)
})

results_df = pd.DataFrame(results)

print("\n========== LightGBM Results ==========")
print(results_df.to_string(index=False))

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [ ]:
import joblib

# Save model
# ==========================================================
joblib.dump(lgb_model, "lightgbm_model(1).pkl")

print("✅ LightGBM saved successfully!")

✅ LightGBM saved successfully!


In [ ]:
import os
import shutil

# Google Drive destination
save_path = "/content/drive/MyDrive/MedAssist AI"
os.makedirs(save_path, exist_ok=True)

shutil.copy("/content/lightgbm_model(1).pkl",
            f"{save_path}/lightgbm_model(1).pkl")

print("model successfully copied to Google Drive!")

model successfully copied to Google Drive!
